# Session 3 — Python and Modelica: the same models, two ways

Guided notebook for Session 3. Canonical lesson: `course/session_03_modelica/lesson.md`.

The notebook runs OpenModelica from Python with `energy_training.omc`: a local `omc` if OpenModelica is installed, otherwise the course Docker image. Each simulation takes a few seconds (the model is translated to C and compiled first).

1. M01 in Modelica, written as one equation, against the Python model of Session 1.
2. What the compiler sees: the flat equations of M01 composed from components.
3. M02 in Modelica (built by `extends` and `redeclare`) against the Python model of Session 2.
4. Python (causal) and Modelica (acausal) side by side.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from energy_training.omc.results import read_result
from energy_training.omc.runner import flatten, simulate
from energy_training.plotting.style import apply_style, degc, hours
from energy_training.simulation import thermal_zone as tz
from energy_training.simulation import zone_rc as zr
from energy_training.utilities import repo_root

apply_style()
ROOT = repo_root()
OUT = ROOT / "output/s03_notebook"  # git-ignored
LIB = ROOT / "modelica/EnergyTraining"

## 1. M01 as one Modelica equation

The model below is the Session 1 energy balance $C\,\dot T = \dot Q - UA\,(T - T_{amb})$, written once as an equation. Nothing says which variable is computed from which: the compiler decides.

In [ ]:
print((LIB / "Examples/S03_ThermalZoneEquations.mo").read_text().split("  annotation")[0])

**Predict:** with the same parameters (C = 1e7 J/K, UA = 200 W/K, illustrative), 4 kW from 10 °C at 0 °C outside, how large will the difference between the Modelica and the Python result be after 72 h?

In [ ]:
run = simulate("EnergyTraining.Examples.S03_ThermalZoneEquations", out_dir=OUT)
m01 = read_result(run.csv)
t = m01.index.to_numpy(dtype=float)
p = tz.ThermalZoneParams()
py = tz.simulate((0, t[-1]), 283.15, 4000.0, 273.15, p, t_eval=t)
exact = tz.analytic_step(t, 283.15, 4000.0, 273.15, p)
print(f"T(72 h): Modelica {degc(m01['T'].iloc[-1]):.4f} degC, Python {degc(py.T[-1]):.4f} degC")
print(f"max |Modelica - exact| = {np.abs(m01['T'].to_numpy() - exact).max():.1e} K")
print(f"max |Python   - exact| = {np.abs(py.T - exact).max():.1e} K")
print(f"max |Modelica - Python| = {np.abs(m01['T'].to_numpy() - py.T).max():.1e} K")

Both results equal the exact solution to within the solver tolerances (OpenModelica's default solver DASSL with tolerance 1e-8 from the `experiment` annotation; SciPy's RK45 with rtol = atol = 1e-8). The remaining differences are integration errors, not model differences.

## 2. What the compiler sees: flat equations

`S03_ThermalZoneTeaching` builds the same M01 from four components and the teaching connector `HeatPort` (temperature `T`, flow variable `Q_flow`). Flattening replaces every component by its equations and every `connect` by **connection equations**. **Predict:** which equations will the node where heater, zone and loss meet produce?

In [ ]:
flat = flatten("EnergyTraining.Examples.S03_ThermalZoneTeaching", out_dir=OUT)
equations = flat.split("equation", 1)[1].rsplit("end ", 1)[0].strip().splitlines()
print("\n".join(e.strip() for e in equations))

The first lines come from `connect`: equal temperatures at each node (potential variables) and heat flow rates that sum to zero (flow variables). The rest are the component equations. Count unknowns and equations:

In [ ]:
declarations = flat.split("equation", 1)[0].splitlines()[1:]
unknowns = [d for d in declarations if d.strip().startswith("Real ")]
print(f"{len(unknowns)} unknowns (Real variables, parameters excluded), {len(equations)} equations")

Equal counts are necessary for a simulation model (a *balanced* model). Most equations are trivial (`zone.port.T = loss.port_a.T`, ...); once the compiler has eliminated them, one differential equation is left, $C\,\dot T = \dot Q - G\,(T - T_{amb})$. The forms of M01 (equation, teaching components, MSL components, and the block diagram `S03_ThermalZoneBlocks`) indeed give the same numbers (`EnergyTraining.Validation.ThermalZoneForms`).

## 3. M02 by `extends` and `redeclare`

`EnergyTraining.Systems.ZoneRC` extends M01 (`Systems.ThermalZone`) and replaces its envelope: the single conductance `EnvelopeUA` becomes `EnvelopeRC` (conductance, capacity, conductance).

In [ ]:
print((LIB / "Systems/ZoneRC.mo").read_text().split("  annotation")[0])

In [ ]:
m02 = read_result(simulate("EnergyTraining.Examples.S03_ZoneRCStep", out_dir=OUT).csv)
t2 = m02.index.to_numpy(dtype=float)
p2 = zr.ZoneRCParams()
py2 = zr.simulate((0, t2[-1]), zr.steady_state(1000.0, 273.15, p2), 4000.0, 273.15, p2, t_eval=t2)
dz = np.abs(m02["zone.T"].to_numpy() - py2.T_z).max()
de = np.abs(m02["zone.T_e"].to_numpy() - py2.T_e).max()
print(f"max |dT_z| = {dz:.1e} K, max |dT_e| = {de:.1e} K")

fig, ax = plt.subplots()
ax.plot(hours(t2), degc(py2.T_z), color="k", label="Python $T_z$")
ax.plot(hours(t2), degc(py2.T_e), color="0.4", label="Python $T_e$")
ax.plot(hours(t2), degc(m02["zone.T"]), "--", label="Modelica $T_z$")
ax.plot(hours(t2), degc(m02["zone.T_e"]), "--", label="Modelica $T_e$")
ax.set(xlabel="time [h]", ylabel="temperature [°C]", title="M02: heater 1 → 4 kW from equilibrium")
ax.legend(loc="lower right");

## 4. Python (causal) and Modelica (acausal)

| | Python (Sessions 1–2) | Modelica (Session 3) |
|---|---|---|
| Model written as | right-hand side $\dot x = f(x, u)$ | equations, in any order and form |
| Who sorts and solves | you | the compiler |
| Reuse | functions, copy the equations | components connected through connectors |
| Change the structure (M01 → M02) | new `rhs` and state vector | `redeclare` one component |
| Energy balance at a node | written by hand | generated by `connect` (flow variables sum to zero) |
| Integration | `solve_ivp` (RK45, BDF, ...) | DASSL by default (Session 4: solvers and settings) |
| Results | NumPy arrays | result file (`.mat` or `.csv`), read here with pandas |

**Exercise G3.1** builds M02 a third way, from MSL components connected by hand, and compares it with Python using `exercises/session_03/s03_g1_compare.py`.